# Data Extraction and Integration Pipeline

This notebook serves as the primary data ingestion engine for our project. It consolidates independent energy, infrastructure, and infrastructure-adoption data tables from three distinct sources: a local flat file, a targeted HTML scrape, and an active government REST API endpoint. Each step handles its own processing, formats structural features, logs raw snapshots, and outputs standardized regional metrics for modeling.

In [ ]:
import os
import pandas as pd
import requests
from bs4 import BeautifulSoup

## Part 1: Processing Local Fuel Station Registries

Our first dataset tracks public vehicle infrastructure metrics nationwide. This block parses the raw tabular records, archives a local baseline snapshot, isolates active public charging ports, cleans missing field variables, and removes duplicate station profiles before updating our processed data files.

In [ ]:
def run_fuel_station_pipeline():
    """
    Data Provenance:
    - Primary Source: U.S. Department of Energy (DOE)
    - Administration: Energy Efficiency and Renewable Energy (EERE)
    - Database: Alternative Fuels Data Center (AFDC) Refueling Registry
    """
    print("Reading raw alternative fuel stations file...")
    # Load raw file download into our primary parsing framework
    data_grid = pd.read_csv("alt_fuel_stations (Oct 8 2025).csv")

    # Log baseline data snapshot before filtering changes
    data_grid.to_csv("data/raw/raw_alt_fuel_stations.csv", index=False)

    # Locate hardware terminal columns for ports
    terminal_columns = [col for col in data_grid.columns if 'EVSE' in col or 'CHARGER' in col.upper()]
    
    # Fill terminal missing records; if operational ('E'), default count to 1
    for current_col in terminal_columns:
        data_grid[current_col] = data_grid.apply(
            lambda row: 1 if (
                (pd.isna(row[current_col]) or str(row[current_col]).strip().upper() in ['N/A', 'NA', '', 'NONE']) 
                and str(row.get('Status Code', '')).upper() == 'E'
            ) else row[current_col], 
            axis=1
        )
        data_grid[current_col] = pd.to_numeric(data_grid[current_col]).fillna(1)

    # Keep open/operational stations and eliminate future planned sites
    data_grid = data_grid[data_grid['Status Code'].astype(str).str.upper() == 'E']
    
    # Clear redundant facility entries to avoid skewing density markers
    data_grid = data_grid.drop_duplicates(subset='Station Name', keep='first')

    # Export clean dataframe layout
    data_grid.to_csv("data/processed/cleaned_alt_fuel_stations.csv", index=False)
    print("Success: Tracked fuel station registries updated.")
    return data_grid

## Part 2: Scraping Light-Duty Vehicle Registrations

This function targets the live market summary matrix hosted on the AFDC tracking site. It reads the raw HTML layout, extracts row components, isolates regional text attributes, strips layout characters, and writes clean integer values to a separate tracking file.

In [ ]:
def run_web_scraper_pipeline():
    """
    Data Provenance:
    - Primary Source: National Renewable Energy Laboratory (NREL) / Experian
    - Metrics Platform: Alternative Fuels Data Center Market Summaries
    - Web Endpoint: https://afdc.energy.gov/data/10962
    """
    print("Fetching live consumer registration profiles...")
    target_url = "https://afdc.energy.gov/data/10962"
    server_response = requests.get(target_url)
    
    # Read document trees to find data structure layers
    document_soup = BeautifulSoup(server_response.text, "html.parser")
    target_table = document_soup.find("table")
    table_rows = target_table.find_all("tr")

    # Map text vectors directly across rows
    regional_markers = [node.get_text(strip=True) for node in table_rows[0].find_all("td")][1:]
    population_counts = [node.get_text(strip=True) for node in table_rows[1].find_all("td")][1:]

    # Package raw text structures into dataframes
    market_dataframe = pd.DataFrame({
        "State": regional_markers,
        "EV_Registrations": population_counts
    })

    # Save raw uncleaned scraper footprint
    market_dataframe.to_csv("data/raw/raw_EV_registrations.csv", index=False)

    # Strip layout commas and formatting punctuation flags before type conversions
    market_dataframe["EV_Registrations"] = market_dataframe["EV_Registrations"].astype(str).str.replace(",", "", regex=True)
    market_dataframe["EV_Registrations"] = market_dataframe["EV_Registrations"].astype(int)

    # Save final cleaned registration data
    market_dataframe.to_csv("data/processed/clean_EV_registrations.csv", index=False)
    print("Success: Scraped market registries updated.")
    return market_dataframe

## Part 3: Querying the Macro Emissions API

Our final component handles the integration of carbon emission data. This engine cycles through the Energy Information Administration (EIA) server pagination rings to parse macro sector responses, flatten raw multi-layered JSON payloads, isolate transportation emission factors, and standardize matching parameters.

In [ ]:
def run_emissions_api_pipeline():
    """
    Data Provenance:
    - Primary Source: U.S. Energy Information Administration (EIA)
    - Information Module: State Energy Data System (SEDS) Macro Metrics
    - Core API Gateway: https://api.eia.gov/v2/co2-emissions/co2-emissions-aggregates/data/
    """
    print("Running remote API loops for carbon trackers...")
    access_key = "VoIcqFbogfFfhhqgduqf7kiRFrcQf5ger24pWmaI"
    query_template = (
        "https://api.eia.gov/v2/co2-emissions/co2-emissions-aggregates/data/"
        f"?api_key={access_key}"
        "&frequency=annual"
        "&data[0]=value"
        "&facets[sectorId][]=TC"
        "&offset={offset}"
        "&length=5000"
    )

    aggregated_records = []
    current_offset = 0

    # Loop pagination parameters until the pipeline hits terminal records
    while True:
        target_endpoint = query_template.format(offset=current_offset)
        api_response = requests.get(target_endpoint)
        payload_json = api_response.json()

        extracted_block = payload_json.get("response", {}).get("data", [])
        if not extracted_block:
            break

        aggregated_records.extend(extracted_block)
        if len(extracted_block) < 5000:
            break
        current_offset += 5000

    emissions_dataframe = pd.DataFrame(aggregated_records)

    # Save raw API file footprint
    emissions_dataframe.to_csv("data/raw/raw_transportation_CO2_emissions.csv", index=False)
    
    # Strip layout noise and symbols from raw strings
    emissions_dataframe["value"] = emissions_dataframe["value"].astype(str).str.replace(r"[^\d\.]", "", regex=True)
    emissions_dataframe["value"] = pd.to_numeric(emissions_dataframe["value"], errors="coerce")

    # Standardize mixed parameters (Billion Btu metrics are updated to Mass Tons)
    if "unit" in emissions_dataframe.columns:
        emissions_dataframe.loc[emissions_dataframe["unit"] == "Billion Btu", "value"] = emissions_dataframe["value"] * 0.00005307
        emissions_dataframe["unit"] = "Million Metric Tons of CO2"

    emissions_dataframe["value"] = emissions_dataframe["value"].fillna(0).astype(float)

    # Save final cleaned API metrics dataframe
    emissions_dataframe.to_csv("data/processed/transportation_CO2_emissions.csv", index=False)
    print("Success: Environmental API registries updated.")
    return emissions_dataframe

## Part 4: Main Execution Block

This cell initializes local file directories if they don't already exist and runs all three ingestion engines sequentially to assemble your local dataset workspace.

In [ ]:
if __name__ == "__main__":
    # Set up our project file layout dynamically
    os.makedirs("data/raw", exist_ok=True)
    os.makedirs("data/processed", exist_ok=True)
    
    print("Beginning complete ingestion sequence...")
    fuel_stations_df = run_fuel_station_pipeline()
    registrations_df = run_web_scraper_pipeline()
    emissions_df = run_emissions_api_pipeline()
    print("Pipeline run complete. Local data workspace populated.")